In [1]:
import datasets

/home/carlos/projects/homeworks/nlp/hw2_template/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = "SlavaYus/tinysms_qa_12m"
dataset = datasets.load_dataset(path)

In [ ]:
import pandas as pd
import json

code = (
"""
def solve() -> int:
    return 54 / 1.1
"""
)
data = [{"code": code}] * 1919

with open("pass1_questions.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [5]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

from transformers import (
    PreTrainedTokenizerFast,
    GPT2Config,
    GPT2LMHeadModel,
)

# ============================================================
# НАСТРОЙКИ
# ============================================================

TOKENIZER_ROWS = 2_000_000   # сколько строк использовать для обучения tokenizer
VOCAB_SIZE = 32_000 
CONTEXT_LENGTH = 1024

# ============================================================
# 1. Берём подвыборку для tokenizer
# ============================================================

tokenizer_dataset = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(min(TOKENIZER_ROWS, len(dataset["train"]))))
)

print(f"Tokenizer train rows: {len(tokenizer_dataset):,}")


# ============================================================
# 2. Iterator без загрузки всего датасета в RAM
# ============================================================

def text_iterator(batch_size=10_000):
    for start in range(0, len(tokenizer_dataset), batch_size):
        batch = tokenizer_dataset[start:start + batch_size]

        for question, code in zip(batch["question"], batch["code"]):
            yield (
                "<|question|>\n"
                + str(question)
                + "\n<|code|>\n"
                + str(code)
                + "\n<|end|>"
            )


# ============================================================
# 3. Обучаем ByteLevel BPE tokenizer с нуля
# ============================================================

tokenizer_backend = Tokenizer(
    BPE(unk_token="<|unk|>")
)

tokenizer_backend.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

tokenizer_backend.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=[
        "<|pad|>",
        "<|unk|>",
        "<|bos|>",
        "<|eos|>",
        "<|question|>",
        "<|code|>",
        "<|end|>",
    ],
)

tokenizer_backend.train_from_iterator(
    text_iterator(),
    trainer=trainer,
    length=len(tokenizer_dataset),
)


# ============================================================
# 4. Оборачиваем tokenizer для Transformers
# ============================================================

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    pad_token="<|pad|>",
    unk_token="<|unk|>",
    bos_token="<|bos|>",
    eos_token="<|eos|>",
)

tokenizer.model_max_length = CONTEXT_LENGTH


# ============================================================
# 5. Создаём ПУСТУЮ GPT-модель
#    НИКАКИХ pretrained weights
# ============================================================

config = GPT2Config(
    vocab_size=len(tokenizer),

    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,

    n_embd=512,
    n_layer=8,
    n_head=8,

    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,

    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = GPT2LMHeadModel(config)


# ============================================================
# 6. Проверяем
# ============================================================

n_params = sum(p.numel() for p in model.parameters())

print()
print("=" * 50)
print(f"Vocab size:      {len(tokenizer):,}")
print(f"Context length:  {CONTEXT_LENGTH}")
print(f"Parameters:      {n_params:,}")
print(f"Parameters:      {n_params / 1e6:.2f}M")
print("=" * 50)


# ============================================================
# 7. Проверяем tokenizer
# ============================================================

example = """<|question|>
Write a Python function that returns the sum of two numbers.
<|code|>
def add(a, b):
    return a + b
<|end|>"""

encoded = tokenizer(example)

print("\nExample:")
print(example)

print("\nToken count:")
print(len(encoded["input_ids"]))

print("\nFirst token ids:")
print(encoded["input_ids"][:30])

print("\nDecoded:")
print(tokenizer.decode(encoded["input_ids"]))


# ============================================================
# 8. Сохраняем tokenizer и пустую модель
# ============================================================

tokenizer.save_pretrained("./code_tokenizer")
model.save_pretrained("./code_model_init")

print("\nSaved:")
print("  ./code_tokenizer")
print("  ./code_model_init")

Tokenizer train rows: 2,000,000




Vocab size:      32,000
Context length:  1024
Parameters:      42,128,384
Parameters:      42.13M

Example:
<|question|>
Write a Python function that returns the sum of two numbers.
<|code|>
def add(a, b):
    return a + b
<|end|>

Token count:
31

First token ids:
[4, 141, 28644, 201, 2057, 9721, 505, 6426, 208, 1274, 218, 824, 2590, 20, 141, 5, 141, 251, 891, 14, 71, 18, 202, 3750, 177, 250, 201, 266, 202, 141]

Decoded:
<|question|>
Write a Python function that returns the sum of two numbers.
<|code|>
def add(a, b):
    return a + b
<|end|>


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.47it/s]


Saved:
  ./code_tokenizer
  ./code_model_init


In [6]:
from itertools import chain

# ============================================================
# НАСТРОЙКИ
# ============================================================

BLOCK_SIZE = 1024


# ============================================================
# 1. Превращаем question + code в текст
# ============================================================

def tokenize_function(batch):

    texts = [
        f"<|question|>\n{question}\n<|code|>\n{code}\n<|end|>"
        for question, code in zip(batch["question"], batch["code"])
    ]

    # EOS добавляем после каждого примера
    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False,
    )

    eos_id = tokenizer.eos_token_id

    encoded["input_ids"] = [
        ids + [eos_id]
        for ids in encoded["input_ids"]
    ]

    encoded["attention_mask"] = [
        mask + [1]
        for mask in encoded["attention_mask"]
    ]

    return encoded


# ============================================================
# 2. Токенизируем весь DatasetDict
# ============================================================

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)


# ============================================================
# 3. PACKING
#
# Было:
#
# example 1 -> 73 tokens
# example 2 -> 184 tokens
# example 3 -> 52 tokens
# ...
#
# Станет:
#
# [================ 1024 tokens ================]
# [================ 1024 tokens ================]
# [================ 1024 tokens ================]
#
# То есть GPU не тратится на padding.
# ============================================================

def group_texts(examples):

    concatenated = {
        key: list(chain.from_iterable(examples[key]))
        for key in examples.keys()
    }

    total_length = len(concatenated["input_ids"])

    # Отбрасываем маленький хвост,
    # который не помещается в полный блок
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE

    result = {
        key: [
            values[i:i + BLOCK_SIZE]
            for i in range(0, total_length, BLOCK_SIZE)
        ]
        for key, values in concatenated.items()
    }

    # Для causal language modeling:
    # модель должна предсказывать следующий токен
    result["labels"] = [
        ids.copy()
        for ids in result["input_ids"]
    ]

    return result


packed_dataset = tokenized.map(
    group_texts,
    batched=True,
    batch_size=1000,
    desc="Packing into 1024-token blocks",
)


# ============================================================
# 4. Проверяем результат
# ============================================================

print()
print("=" * 60)
print("TRAIN BLOCKS:", len(packed_dataset["train"]))
print("TEST BLOCKS: ", len(packed_dataset["test"]))
print("=" * 60)

print("\nПервый блок:")

print("input_ids:", len(packed_dataset["train"][0]["input_ids"]))
print(
    "attention_mask:",
    len(packed_dataset["train"][0]["attention_mask"]),
)
print("labels:", len(packed_dataset["train"][0]["labels"]))

print("\nНачало первого блока:\n")

print(
    tokenizer.decode(
        packed_dataset["train"][0]["input_ids"][:300]
    )
)

Packing into 1024-token blocks: 100%|██████████| 1919/1919 [00:00<00:00, 49103.62 examples/s]


TRAIN BLOCKS: 1398581
TEST BLOCKS:  93

Первый блок:
input_ids: 1024
attention_mask: 1024
labels: 1024

Начало первого блока:

<|question|>
Mark has 10 crayons. He gives 2 crayons to his younger sister and loses another 4 while he was playing. How many crayons does Mark have left?
<|code|>
def solve() -> int:
    crayonsTotal = 10
    crayonsGiven = 2
    crayonsLost = 4
    crayonsLeft = crayonsTotal - crayonsGiven - crayonsLost
    result = crayonsLeft

    return result

<|end|><|eos|><|question|>
A group of tourists went on a hike. They hiked for 5 hours and took a 30 minute break. If they hiked in total for 10 miles, what was their average speed in miles/hour?
<|code|>
def solve() -> int:
    total_time = 5.5 #hours
    total_distance_travelled = 10 #miles
    average_speed = total_distance_travelled / total_time
    result = average_speed

    return result

<|end|><|eos|><|question|>
A ball is kicked and its height above the ground is given by the equation h(t) = -16t^2 + 32t +

In [7]:
import os
import math
import torch    

from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm


# ============================================================
# НАСТРОЙКИ
# ============================================================

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

NUM_EPOCHS = 1
WARMUP_RATIO = 0.02

LOG_EVERY = 20
SAVE_EVERY = 2000

CHECKPOINT_DIR = "./checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )


# ============================================================
# BF16 / FP16
# ============================================================

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

dtype = torch.bfloat16 if use_bf16 else torch.float16

print("Mixed precision:", dtype)


# ============================================================
# DATASET -> PYTORCH
# ============================================================

train_dataset = packed_dataset["train"]

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)


# ============================================================
# MODEL
# ============================================================

model = model.to(device)

# Экономим VRAM
model.gradient_checkpointing_enable()

# При gradient checkpointing cache надо отключить
model.config.use_cache = False


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)


# ============================================================
# КОЛИЧЕСТВО ШАГОВ
# ============================================================

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)

total_steps = updates_per_epoch * NUM_EPOCHS

warmup_steps = int(
    total_steps * WARMUP_RATIO
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print()
print("Train batches:", len(train_loader))
print("Optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)

print(
    "Effective batch size:",
    BATCH_SIZE * GRAD_ACCUM_STEPS
)


# ============================================================
# GRAD SCALER
#
# Нужен для FP16.
# Для BF16 scaler не нужен.
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda" and not use_bf16)
)


# ============================================================
# TRAIN
# ============================================================

model.train()

global_step = 0
running_loss = 0.0

optimizer.zero_grad(set_to_none=True)


for epoch in range(NUM_EPOCHS):

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for batch_idx, batch in enumerate(progress_bar):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )


        # ====================================================
        # FORWARD
        # ====================================================

        with torch.autocast(
            device_type=device.type,
            dtype=dtype,
            enabled=(device.type == "cuda"),
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss

            # gradient accumulation
            loss = loss / GRAD_ACCUM_STEPS


        # ====================================================
        # BACKWARD
        # ====================================================

        scaler.scale(loss).backward()

        running_loss += loss.item() * GRAD_ACCUM_STEPS


        # ====================================================
        # OPTIMIZER STEP
        # ====================================================

        if (
            (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
            or batch_idx + 1 == len(train_loader)
        ):

            # сначала возвращаем нормальный scale градиентов
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()

            global_step += 1


            # ================================================
            # LOGGING
            # ================================================

            if global_step % LOG_EVERY == 0:

                avg_loss = (
                    running_loss
                    / (LOG_EVERY * GRAD_ACCUM_STEPS)
                )

                try:
                    perplexity = math.exp(avg_loss)
                except OverflowError:
                    perplexity = float("inf")

                lr = scheduler.get_last_lr()[0]

                progress_bar.set_postfix(
                    loss=f"{avg_loss:.4f}",
                    ppl=f"{perplexity:.2f}",
                    lr=f"{lr:.2e}",
                )

                running_loss = 0.0


            # ================================================
            # CHECKPOINT
            # ================================================

            if global_step % SAVE_EVERY == 0:

                save_path = os.path.join(
                    CHECKPOINT_DIR,
                    f"step_{global_step}"
                )

                model.save_pretrained(
                    save_path,
                    safe_serialization=True
                )

                tokenizer.save_pretrained(
                    save_path
                )

                torch.save(
                    {
                        "global_step": global_step,
                        "epoch": epoch,
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                    },
                    os.path.join(
                        save_path,
                        "training_state.pt"
                    )
                )

                print(
                    f"\nCheckpoint saved: {save_path}"
                )


# ============================================================
# СОХРАНЯЕМ ФИНАЛЬНУЮ МОДЕЛЬ
# ============================================================

FINAL_DIR = "./code_model_final"

model.save_pretrained(
    FINAL_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(
    FINAL_DIR
)

print()
print("=" * 60)
print("TRAINING FINISHED")
print("Model saved to:", FINAL_DIR)
print("=" * 60)

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GB
Mixed precision: torch.bfloat16

Train batches: 349645
Optimizer steps: 43706
Warmup steps: 874
Effective batch size: 32


Epoch 1/1:   2%|▏         | 7470/349645 [06:43<5:08:04, 18.51it/s, loss=2.1132, lr=3.00e-04, ppl=8.27]    


KeyboardInterrupt: 